### Завдання 1.
Звантажити та відкрити датасет: Individual Household Electric Power Consumption Dataset. Здійснити data cleaning.

In [3]:
import pandas as pd
import numpy as np
import timeit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from IPython.display import display
df_household = pd.read_csv('household_power_consumption.txt', sep=';', na_values=['?'], low_memory=False)
df_household.dropna(inplace=True)
print("-" * 50)
print(f"Очищення успішно завершено! Загальна кількість валідних записів: {len(df_household)}")
display(df_household.head(4))

--------------------------------------------------
Очищення успішно завершено! Загальна кількість валідних записів: 2049280


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0


### Завдання 2.1. 
Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.

In [11]:
def filter_high_power(df):
    return df[df['Global_active_power'] > 5.0]
print("Потужність > 5 кВт")
start_time = timeit.default_timer()
df_high_power = filter_high_power(df_household)
execution_time = timeit.default_timer() - start_time
display(df_high_power.head(3))

Потужність > 5 кВт


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0


### Завдання 2.2. 
Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильник споживають більше, ніж бойлер та кондиціонер.

In [12]:
def filter_intensity_and_appliances(df):
    cond1 = df['Global_intensity'].between(19.0, 20.0)
    cond2 = df['Sub_metering_2'] > df['Sub_metering_3']
    return df[cond1 & cond2]
print("Струм 19-20А та Група 2 > Група 3")
start_time = timeit.default_timer()
df_appliances = filter_intensity_and_appliances(df_household)
execution_time = timeit.default_timer() - start_time
print(f"Знайдено записів: {len(df_appliances)}")
display(df_appliances.head(3))

Струм 19-20А та Група 2 > Група 3
Знайдено записів: 2509


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0


### Завдання 2.3. 
Обрати випадковим чином 500000 записів (без повторів), для них обчислити середні величини усіх 3-х груп споживання електричної енергії.

In [13]:
def sample_and_average(df):
    df_sample = df.sample(n=500000, replace=False, random_state=42)
    means = df_sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return means
print("Середнє для 500 000 випадкових записів")
start_time = timeit.default_timer()
average_consumption = sample_and_average(df_household)
execution_time = timeit.default_timer() - start_time
print("\nСереднє споживання по групах:")
print(average_consumption)

Середнє для 500 000 випадкових записів

Середнє споживання по групах:
Sub_metering_1    1.119258
Sub_metering_2    1.308912
Sub_metering_3    6.452950
dtype: float64


### Завдання 2.4. 
Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [15]:
def filter_evening_complex(df):
    cond1 = df['Time'] >= '18:00:00'
    cond2 = df['Global_active_power'] > 6.0
    subset = df[cond1 & cond2]
    cond3 = (subset['Sub_metering_2'] > subset['Sub_metering_1']) & \
            (subset['Sub_metering_2'] > subset['Sub_metering_3'])
    filtered = subset[cond3]
    half_index = len(filtered) // 2
    first_half = filtered.iloc[:half_index:3]
    second_half = filtered.iloc[half_index::4]
    return pd.concat([first_half, second_half])
print("Вечірнє комплексне фільтрування")
start_time = timeit.default_timer()
df_evening_final = filter_evening_complex(df_household)
execution_time = timeit.default_timer() - start_time
print(f"Відібрано фінальних записів: {len(df_evening_final)}")
display(df_evening_final.head(5))

Вечірнє комплексне фільтрування
Відібрано фінальних записів: 310


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0
17498,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0
17501,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0


### Завдання 3. 
Пронормувати та стандартизувати вибраний датасет.

In [17]:
numeric_features = [
    'Global_active_power', 'Global_reactive_power', 'Voltage', 
    'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
]
start_time = timeit.default_timer()
df_numeric_copy = df_household[numeric_features].copy()
min_max_scaler = MinMaxScaler()
std_scaler = StandardScaler()
df_normalized = pd.DataFrame(
    min_max_scaler.fit_transform(df_numeric_copy), 
    columns=numeric_features
)
df_standardized = pd.DataFrame(
    std_scaler.fit_transform(df_numeric_copy), 
    columns=numeric_features
)
execution_time = timeit.default_timer() - start_time
display(df_normalized.head(4))
print("\n2. Стандартизовані дані (StandardScaler, середнє = 0, дисперсія = 1):")
display(df_standardized.head(4))

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,0.374796,0.300719,0.376090,0.377593,0.0,0.0125,0.548387
1,0.478363,0.313669,0.336995,0.473029,0.0,0.0125,0.516129
2,0.479631,0.358273,0.326010,0.473029,0.0,0.0250,0.548387
3,0.480898,0.361151,0.340549,0.473029,0.0,0.0125,0.548387



2. Стандартизовані дані (StandardScaler, середнє = 0, дисперсія = 1):


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2.955077,2.610721,-1.851816,3.098789,-0.182337,-0.051274,1.249421
1,4.037085,2.770406,-2.225274,4.133800,-0.182337,-0.051274,1.130897
2,4.050326,3.320432,-2.330213,4.133800,-0.182337,0.120487,1.249421
3,4.063567,3.355917,-2.191324,4.133800,-0.182337,-0.051274,1.249421


### Завдання 4. 
Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.

In [20]:
col_power = 'Global_active_power'
col_intensity = 'Global_intensity'
pearson_corr = df_household[col_power].corr(df_household[col_intensity], method='pearson')
spearman_corr = df_household[col_power].corr(df_household[col_intensity], method='spearman')
print(f"🔹 Коефіцієнт Пірсона:  {pearson_corr:.5f}")
print(f"🔹 Коефіцієнт Спірмена: {spearman_corr:.5f}")

🔹 Коефіцієнт Пірсона:  0.99889
🔹 Коефіцієнт Спірмена: 0.99537


### Завдання 5. 
Провести One Hot Encoding категоріального атрибута. Створюю категоріальний атрибут "День тижня" з колонки `Date` та застосуємо до нього OHE

In [29]:
def encode_days_final(df, date_column='Date'):
    temp_df = df[[date_column]].copy()
    temp_df[date_column] = pd.to_datetime(temp_df[date_column], format='%d/%m/%Y')
    temp_df['Day_of_Week'] = temp_df[date_column].dt.day_name()
    temp_df['Is_Working_Day'] = temp_df[date_column].dt.dayofweek < 5
    temp_df['Is_Weekend'] = temp_df[date_column].dt.dayofweek >= 5
    return temp_df
df_display = encode_days_final(df_household)
print("Результат перевірки робочих та вихідних днів")
display(df_display.head(5))

Результат перевірки робочих та вихідних днів


,Date,Day_of_Week,Is_Working_Day,Is_Weekend
0,2006-12-16,Saturday,False,True
1,2006-12-16,Saturday,False,True
2,2006-12-16,Saturday,False,True
3,2006-12-16,Saturday,False,True
4,2006-12-16,Saturday,False,True
